In [ ]:
import os
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))
from paths import PROCESSED, RESULTS, savefig
from pypsa_helpers import solve_scenario, require_optimal, total_co2, total_system_cost, scale_capital_cost

import pypsa
import pandas as pd
import matplotlib.pyplot as plt

Sensitivity: Combined H2 Technology Cost Reduction (100% CO2 Reduction Scenario)

Both H2 electrolysis and fuel cell capital costs reduced together, at realistic levels only
(25% and 50% reduction - i.e. 75% and 50% of baseline cost). 75%/100% reduction is left out
here since assuming *both* technologies simultaneously approach zero cost isn't a realistic
joint cost projection, unlike testing each one individually to its extreme as a bounding case
(see `09_sensitivity_electrolysis_cost.ipynb` and `12_sensitivity_fuel_cell_cost.ipynb`).

Parameterized by the `H2_COST_FRACTION` environment variable (defaults to `1.0`).

In [ ]:
cost_fraction = float(os.environ.get("H2_COST_FRACTION", "1.0"))
assert 0.5 <= cost_fraction <= 1.0  # only the realistic 25%/50% reduction range is in scope here
print("H2 technology (electrolysis + fuel cell) capital cost fraction:", cost_fraction)

In [ ]:
n = pypsa.Network(str(PROCESSED / "pypsa_network_base.nc"))
cost_scale = n.meta["cost_scale"]

n = scale_capital_cost(n, carrier="H2 electrolysis", fraction=cost_fraction)
n = scale_capital_cost(n, carrier="H2 fuel cell", fraction=cost_fraction)
n = solve_scenario(n, co2_limit=0)

In [ ]:
print("status:", n.meta["status"], "| termination condition:", n.meta["termination_condition"])

require_optimal(n)

total_cost_eur = total_system_cost(n, cost_scale=cost_scale)
total_co2_t = total_co2(n)
fuel_cell_capacity_mw = n.links.loc[n.links.carrier == "H2 fuel cell", "p_nom_opt"].sum()
electrolysis_capacity_mw = n.links.loc[n.links.carrier == "H2 electrolysis", "p_nom_opt"].sum()

print(f"H2 technology cost fraction:    {cost_fraction:.0%}")
print(f"total annual system cost:       {total_cost_eur:,.0f} EUR/yr")
print(f"total CO2 emissions:            {total_co2_t:,.0f} t/yr")
print(f"fuel cell capacity built:       {fuel_cell_capacity_mw:,.0f} MW")
print(f"electrolysis capacity built:    {electrolysis_capacity_mw:,.0f} MW")

In [ ]:
pct_label = f"{round(cost_fraction * 100)}pct"
n.export_to_netcdf(str(PROCESSED / f"pypsa_sensitivity_combined_h2_{pct_label}.nc"))